In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import other_files_location, intermediate_files_location



# Loading NC Coherent 1g simulation file

In [ ]:
# downloaded from /pnfs/uboone/persistent/users/markross/Jan2022_gLEE_files/UniqDir/vertex_NCSingleCoherent_Run123_BatchB_v50.0.uniq.root
# downloaded from /exp/uboone/data/users/gge/SinglePhotonAnalysis/FluxEventweightAssignment/preTopo_vertex_NCSingleCoherent_Run123_TestSplit_v50.0_OrderedSample_NCDelta.uniq.root
#f = uproot.open(other_files_location + "/vertex_NCSingleCoherent_Run123_BatchB_v50.0.uniq.root")["singlephotonana/vertex_tree"]
f = uproot.open(other_files_location + "/preTopo_vertex_NCSingleCoherent_Run123_TestSplit_v50.0_OrderedSample_NCDelta.uniq.root")["singlephotonana/vertex_tree"]
daughters_pdg = f["mctruth_daughters_pdg"].array(library="np")
daughters_E = f["mctruth_daughters_E"].array(library="np")
daughters_pz = f["mctruth_daughters_pz"].array(library="np")

nc_coherent_gamma_energy = []
nc_coherent_gamma_costheta = []

for i in range(len(daughters_pdg)):
    if daughters_pdg[i][0] != 22:
        raise Exception("PROBLEM! First daughter particle isn't a photon in the nc coherent 1g simulation", daughters_pdg[i])
    nc_coherent_gamma_energy.append(daughters_E[i][0] * 1000)
    nc_coherent_gamma_costheta.append(daughters_pz[i][0] / daughters_E[i][0])
    

bins = [np.linspace(-1, 1, 100), np.linspace(0, 1000, 100)]

# Compare with Figure 1 of https://arxiv.org/pdf/2502.06091
cmap = mpl.colormaps.get_cmap("viridis").copy()
cmap.set_under("white")   # color for values < vmin
plt.figure(figsize=(10, 5))
plt.hist2d(nc_coherent_gamma_costheta, nc_coherent_gamma_energy, bins=bins, cmap=cmap, vmin=1e-6)
plt.colorbar()
plt.xlabel("Cosine of Theta")
plt.ylabel("Energy (MeV)")
plt.title("NC Coherent Gamma Energy and Costheta")
plt.show()


In [ ]:
f = uproot.open(other_files_location + "/vertex_NCSingleCoherent_Run123_BatchB_v50.0.uniq.root")

np.sum(f["singlephotonana/run_subrun_tree"]["subrun_pot"].array(library="np"))

# Loading Isotropic 1g simulation events

In [ ]:
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")

iso1g_df = all_df.filter(pl.col("filetype") == "isotropic_one_gamma_overlay")

iso1g_df = iso1g_df.select(
    "run",
    "subrun",
    "event",
    "wc_true_leading_shower_energy",
    "wc_true_leading_shower_costheta",
    "wc_net_weight",
    "wc_truth_inFV",
    "wc_truth_vtxX",
    "wc_truth_vtxY",
    "wc_truth_vtxZ",
).collect()

plt.figure(figsize=(10, 5))
plt.hist2d(
    iso1g_df["wc_true_leading_shower_costheta"].to_numpy(),
    iso1g_df["wc_true_leading_shower_energy"].to_numpy(),
    bins=bins, cmap="viridis", norm=mpl.colors.LogNorm(),
    weights=iso1g_df["wc_net_weight"].to_numpy(),
)
plt.colorbar()
plt.xlabel("Cosine of Theta")
plt.ylabel("Energy (MeV)")
plt.title("Isotropic 1g Gamma Energy and Costheta (weighted)")
plt.show()


In [ ]:
import matplotlib.colors as mcolors

# 2D reweighting: iso1g → NC coherent 1g shape
# Bin edges with overflow
bins_costheta = np.linspace(-1, 1, 101)
bins_energy   = np.concatenate([np.linspace(0, 2000, 101), [1e6]])

iso1g_costheta = iso1g_df["wc_true_leading_shower_costheta"].to_numpy()
iso1g_energy   = iso1g_df["wc_true_leading_shower_energy"].to_numpy()
iso1g_w        = iso1g_df["wc_net_weight"].to_numpy()

H_coherent, _, _ = np.histogram2d(
    nc_coherent_gamma_costheta, nc_coherent_gamma_energy,
    bins=[bins_costheta, bins_energy],
)
H_iso1g, _, _ = np.histogram2d(
    iso1g_costheta, iso1g_energy,
    bins=[bins_costheta, bins_energy],
    weights=iso1g_w,
)

# Weight per bin: ratio of coherent to iso1g (weighted)
weight_ratios = np.where(H_iso1g > 0, H_coherent / H_iso1g, 0.0)

# Assign weight to each iso1g event
i_cos = np.clip(np.digitize(iso1g_costheta, bins_costheta) - 1, 0, len(bins_costheta) - 2)
i_E   = np.clip(np.digitize(iso1g_energy,   bins_energy)   - 1, 0, len(bins_energy)   - 2)
event_weights = weight_ratios[i_cos, i_E]

iso1g_df = iso1g_df.with_columns(
    pl.Series("coherent_1g_weight_nopot", event_weights)
)

# After-reweighting histogram for validation
H_iso1g_after, _, _ = np.histogram2d(
    iso1g_costheta, iso1g_energy,
    bins=[bins_costheta, bins_energy],
    weights=iso1g_w * event_weights,
)

# Events with positive weight, unweighted histogram
H_iso1g_after_unweighted, _, _ = np.histogram2d(
    iso1g_costheta, iso1g_energy,
    bins=[bins_costheta, bins_energy],
    weights=np.array(iso1g_w * event_weights > 0),
)



# ── Diagnostic plots ──────────────────────────────────────────────────────────
vmax = max(H_coherent.max(), H_iso1g.max())
vmin = vmax * 1e-3 if vmax > 0 else 1e-10

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, H, title in [
    (axes[0], H_iso1g,       "Iso 1g (wc_net_weight weighted)\n[before coherent_1g_weight]"),
    (axes[1], H_coherent,    "NC Coherent 1g simulation (target)"),
    (axes[2], H_iso1g_after, "Iso 1g after coherent_1g_weight"),
]:
    H_plot = np.where(H > 0, H, np.nan)
    im = ax.pcolormesh(bins_costheta, bins_energy, H_plot.T,
                       cmap='plasma', shading='flat',
                       norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
    plt.colorbar(im, ax=ax, label='Events / bin')
    ax.set_xlabel(r"cos $\theta$")
    ax.set_ylabel("Energy [MeV]")
    ax.set_xlim(-1, 1)
    ax.set_ylim(0, 2100)
    ax.set_title(title)
fig.suptitle("2D (cos θ, energy) distribution before and after coherent reweighting")
fig.tight_layout()
plt.show()

vmax = H_iso1g_after_unweighted.max()
vmin = vmax * 1e-3 if vmax > 0 else 1e-10

plt.figure()
plt.pcolormesh(bins_costheta, bins_energy, H_iso1g_after_unweighted.T, cmap='plasma', shading='flat', norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
plt.colorbar()
plt.xlabel(r"cos $\theta$")
plt.ylabel("Energy [MeV]")
plt.title("Iso 1g after coherent_1g_weight (unweighted)\n(used for dedicated Coherent 1g BDT training)")
plt.xlim(-1, 1)
plt.ylim(0, 2100)
plt.show()

# 1D histogram of the weight of coherent_1g_weight_nopot

plt.figure()
plt.hist(iso1g_df["coherent_1g_weight_nopot"].to_numpy(), bins=100, range=(0, 10), density=True)
plt.xlabel("Weight")
plt.ylabel("Events / bin")
plt.title("1D histogram of the weight of coherent_1g_weight_nopot")
plt.yscale("log")
plt.show()

print("total number of Iso1g events:", len(iso1g_df))
print("num Iso1g events with positive weight:", np.sum(iso1g_df["coherent_1g_weight_nopot"].to_numpy() > 0))
print("num Iso1g events with zero weight:", np.sum(iso1g_df["coherent_1g_weight_nopot"].to_numpy() == 0))


In [ ]:
# Random sampling to make the unweighted 2D histogram match the shape of the weighted
# "Iso1g after coherent_1g_weight" distribution, without any event weights.
# Strategy: per-bin acceptance probability proportional to target_shape / current_unweighted_shape,
# with a global scale factor chosen via binary search to hit the target kept fraction.
# Bins that would need p > 1.0 are capped at 1.0 (slight local shape loss, unavoidable).

target_fraction = 0.50  # fraction of positive-weight events to keep

# Normalize target (weighted) and current (unweighted) to densities
total_after      = H_iso1g_after.sum()
total_unweighted = H_iso1g_after_unweighted.sum()

H_after_norm      = H_iso1g_after / total_after           if total_after      > 0 else H_iso1g_after
H_unweighted_norm = H_iso1g_after_unweighted / total_unweighted if total_unweighted > 0 else H_iso1g_after_unweighted

# Per-bin ratio: target / current (unnormalized — will be scaled below)
with np.errstate(divide='ignore', invalid='ignore'):
    ratio = np.where(H_unweighted_norm > 0, H_after_norm / H_unweighted_norm, 0.0)

# Assign raw (unscaled) per-event probabilities
mask_positive = iso1g_df["coherent_1g_weight_nopot"].to_numpy() > 0

i_cos_all = np.clip(np.digitize(iso1g_costheta, bins_costheta) - 1, 0, len(bins_costheta) - 2)
i_E_all   = np.clip(np.digitize(iso1g_energy,   bins_energy)   - 1, 0, len(bins_energy)   - 2)

event_prob_raw = np.where(mask_positive, ratio[i_cos_all, i_E_all], 0.0)

# Binary search for scale factor s so that sum(min(1, s*p)) == target_kept
# lo=0 keeps nothing; hi = 1/min_positive keeps everything (all probs >= 1)
n_positive  = int(mask_positive.sum())
target_kept = target_fraction * n_positive
min_raw     = event_prob_raw[event_prob_raw > 0].min()
lo, hi = 0.0, 1.0 / min_raw
for _ in range(60):
    mid = (lo + hi) / 2
    expected = np.sum(np.minimum(1.0, mid * event_prob_raw))
    if expected < target_kept:
        lo = mid
    else:
        hi = mid
s = (lo + hi) / 2

event_keep_prob = np.minimum(1.0, s * event_prob_raw)

# Randomly accept events
rng = np.random.default_rng(seed=42)
keep_mask = rng.random(len(iso1g_df)) < event_keep_prob

iso1g_df = iso1g_df.with_columns(
    pl.Series("coherent_1g_keep", keep_mask)
)

print(f"Target fraction: {target_fraction:.2f}  ({target_kept:,.0f} events)")
print(f"Events kept:     {keep_mask.sum():,} / {n_positive:,}  ({keep_mask.sum() / n_positive:.3f})")

# Unweighted histogram of kept events for validation
H_sampled, _, _ = np.histogram2d(
    iso1g_costheta[keep_mask], iso1g_energy[keep_mask],
    bins=[bins_costheta, bins_energy],
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, H, title in [
    (axes[0], H_iso1g_after_unweighted, "Iso 1g after coherent_1g_weight\n(unweighted, before sampling)"),
    (axes[1], H_iso1g_after,            "Iso 1g after coherent_1g_weight\n(weighted — target shape)"),
    (axes[2], H_sampled,                f"Iso 1g after random sampling\n(unweighted, {target_fraction:.0%} target)"),
]:
    H_plot = np.where(H > 0, H, np.nan)
    vmax = np.nanmax(H_plot)
    vmin = vmax * 1e-3
    im = ax.pcolormesh(bins_costheta, bins_energy, H_plot.T,
                       cmap='plasma', shading='flat',
                       norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
    plt.colorbar(im, ax=ax, label='Events / bin')
    ax.set_xlabel(r"cos $\theta$")
    ax.set_ylabel("Energy [MeV]")
    ax.set_xlim(-1, 1)
    ax.set_ylim(0, 2100)
    ax.set_title(title)
fig.suptitle("Random sampling to match NC coherent shape without event weights")
fig.tight_layout()
plt.show()


In [ ]:
# POT scaling factor

num_coherent_simulated_events = len(nc_coherent_gamma_energy)
current_normalizing_POT = 2.098e19 + 4.038e19 # for run 4a and run 4b open data

# see discussion with Mark on slack 2026_02_24
reference_nc_coherent_pot = 2.22242e+24
reference_nc_coherent_num_events = 16139
train_test_fudge_factor = 0.5 # guessing, to make the next numbers match better


# see https://arxiv.org/abs/2502.06091
coherent_eff_topological = 0.281
coherent_eff_rel_to_topological = 0.3909
coherent_num_sel = 1.1
coherent_num_fv_687e20 = coherent_num_sel / (coherent_eff_topological * coherent_eff_rel_to_topological)
print(f"Num NC coherent 1g events simulated in FV in 6.87e20 POT from paper: {coherent_num_fv_687e20}")

iso1g_df = iso1g_df.with_columns(
    pl.Series("coherent_1g_weight", iso1g_df["coherent_1g_weight_nopot"] * reference_nc_coherent_num_events / num_coherent_simulated_events
                                                                         * current_normalizing_POT / reference_nc_coherent_pot
                                                                         * train_test_fudge_factor)
)

iso1g_fv_df = iso1g_df.filter(pl.col("wc_truth_inFV") == 1)
iso1g_fv_df = iso1g_df.filter(
    (10.0 < pl.col("wc_truth_vtxX")) & (pl.col("wc_truth_vtxX") < 246.4) &
    (-101.5 < pl.col("wc_truth_vtxY")) & (pl.col("wc_truth_vtxY") < 101.5) &
    (10.0 < pl.col("wc_truth_vtxZ")) & (pl.col("wc_truth_vtxZ") < 986.8)
)

df_coherent_num_fv_687e20 = iso1g_fv_df["coherent_1g_weight"].sum() * 6.87e20 / current_normalizing_POT

print(f"Num NC coherent 1g events simulated in FV in 6.87e20 POT from our df: {df_coherent_num_fv_687e20}")


In [ ]:
# Save per-event coherent 1g reweighting weights.
# Only events with coherent_1g_weight > 0 are saved (iso1g events in empty bins are excluded).

weights_to_save = iso1g_df.filter(
    pl.col("coherent_1g_weight") > 0
).select([
    "run",
    "subrun",
    "event",
    "coherent_1g_weight",
    "coherent_1g_keep",
])

weights_to_save.write_parquet(f"{intermediate_files_location}/coherent_1g_reweighting_weights.parquet")

print(f"Saved coherent_1g_reweighting_weights.parquet")
print(f"  {len(weights_to_save):,} / {len(iso1g_df):,} events have valid weights")
